# Part 2 – Curated Dataset

In [ ]:
# Standard library
from pathlib import Path
import sys

# Add project root to Python path
PROJECT_ROOT = Path().resolve().parents[1]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))


In [ ]:
# Third-party libraries
import pandas as pd
from IPython.display import display
from IPython.core.interactiveshell import InteractiveShell

from src.myh_pipeline.config import (
    BASELINE_YEARS,
    RAW_DATA_PATH,
    TARGET_COLUMNS,
    CURATED_DATA_PATH
)

from src.myh_pipeline.load import (
    load_excel,
    check_schema,
)

from src.myh_pipeline.clean import (
    clean_column_name,
    clean_string_values,
)

from src.myh_pipeline.harmonize import (
    harmonize_schema,
)

from src.myh_pipeline.validate import (
    build_validation_summary,
)

from src.myh_pipeline.enrich import (
    enrich_dataset,
)

In [ ]:
# Jupyter display settings
InteractiveShell.ast_node_interactivity = "all"

## Step 1: Load and Combine Raw Excel Files

The raw data is stored as separate Excel files for different years (2022–2025).

In this step, I load all Excel files from the raw data directory, add a year column based on the filename, and combine them into a single dataframe for further cleaning and harmonization.

Combining the datasets into one dataframe makes it easier to:
- inspect structural differences between years
- identify inconsistencies
- harmonize column names and data types
- perform unified cleaning and transformation

In [2]:
# Excel source files
excel_files = RAW_DATA_PATH.glob("*.xlsx")

dfs = {}

for file in excel_files:
    year = int(file.stem[-4:])
    if year not in BASELINE_YEARS:
        continue
    dfs[year] = load_excel(file)


Loading file: resultat-ansokningsomgang-2022.xlsx
Loading file: resultat-ansokningsomgang-2023.xlsx
Loading file: resultat-ansokningsomgang-2024.xlsx
Loading file: resultat-ansokningsomgang-2025.xlsx


## Step 2: Comparing Table Structures Across Years

To prepare for harmonization, I inspected the structure of Tabell 3 across all years.

The inspection included:
- dataset dimensions
- column names
- structural differences between years

The comparison revealed that the 2022 dataset contains fewer columns than the datasets from 2023–2025.

This indicates that schema harmonization is required before the datasets can be combined into a unified curated dataset.

In [3]:
# Inspect each dataframe
schema_df = check_schema(dfs)
schema_df

,source_year,shape,columns
0,2022,"(1207, 19)","[Utbildningsområde, Utbildningsnamn, Beslut, D..."
1,2023,"(1258, 31)","[Utbildningsområde, SUN5 inriktning, SUN5 inri..."
2,2024,"(1272, 31)","[Utbildningsområde, SUN5 inriktning, SUN5 inri..."
3,2025,"(1184, 31)","[Utbildningsområde, SUN5 inriktning, SUN5 inri..."


### Schema Comparison Summary

The historical Excel files shared several core business concepts, but their structures were not fully consistent across years.

The main differences were:

- different header row positions across years
- slightly different column names
- inconsistent naming conventions
- some variation in available columns

Because of this, I created a harmonized target schema and mapped the source columns into a consistent structure.

## Step 3: Define Target Schema

Based on the schema comparison, a common target schema was defined for the curated dataset.

### Column Mapping Rationale

The source Excel files used slightly different column names across years for the same business concepts.

Examples:

| Original Columns | Harmonized Column |
|---|---|
| Utbildningsanordnare administrativ enhet | utbildningsanordnare |
| Studietakt % | studietakt_procent |
| Beslut | beslut |
| Studieform | studieform |

The harmonization was based on semantic meaning rather than exact text matching.

In addition, normalized English analytical columns such as `decision_normalized` and `study_form_normalized` were added to support downstream API filtering and analytics use cases while preserving the original Swedish source terminology.

In [4]:
print(f"Target column count: {len(TARGET_COLUMNS)}")

pd.DataFrame({"target_columns": TARGET_COLUMNS})

Target column count: 20


,target_columns
0,source_year
1,source_file
2,source_sheet
3,diarienummer
4,utbildningsnamn
5,utbildningsomrade
6,beslut
7,decision_normalized
8,kommun
9,lan


## Step 4: Column Standardization & Basic Cleaning

Column names were standardized before harmonization to reduce technical differences between years.

The cleaning rules included:

- converting all column names to lowercase
- removing leading and trailing spaces
- replacing Swedish characters with ASCII equivalents
- replacing spaces and special characters with underscores
- removing parentheses and percentage symbols where needed

This made it easier to compare columns across years and apply a consistent column mapping.

In [5]:
# clean all years
standardized_dfs = {}

for year, df in dfs.items():
    df = df.copy()

    # clean column names
    df.columns = [clean_column_name(col) for col in df.columns]

    # clean string values
    df = clean_string_values(df)

    standardized_dfs[year] = df


# check result
schema_df = check_schema(standardized_dfs)
schema_df

,source_year,shape,columns
0,2022,"(1207, 19)","[utbildningsomrade, utbildningsnamn, beslut, d..."
1,2023,"(1258, 31)","[utbildningsomrade, sun5_inriktning, sun5_inri..."
2,2024,"(1272, 31)","[utbildningsomrade, sun5_inriktning, sun5_inri..."
3,2025,"(1184, 31)","[utbildningsomrade, sun5_inriktning, sun5_inri..."


## Step 5: Schema Harmonization

After standardizing the column names, the datasets still contained structural differences between years. In particular, the 2022 dataset included fewer columns than the datasets from 2023–2025.

To create a unified curated dataset, the schemas were harmonized into a common target structure.

The harmonization process included:

- aligning all datasets to the predefined target schema
- adding missing columns with null values
- keeping only relevant columns
- ensuring a consistent column order across all years
- adding source metadata columns for traceability

This step ensures that all yearly datasets can be safely combined into a single curated dataset.

In [6]:
# harmonize all dataframes
harmonized_dfs = {}
for year, df in standardized_dfs.items():
    harmonized_df = harmonize_schema(df=df, target_columns=TARGET_COLUMNS)

    harmonized_dfs[year] = harmonized_df


schema_df = check_schema(harmonized_dfs)
schema_df


,source_year,shape,columns
0,2022,"(1207, 20)","[source_year, source_file, source_sheet, diari..."
1,2023,"(1258, 20)","[source_year, source_file, source_sheet, diari..."
2,2024,"(1272, 20)","[source_year, source_file, source_sheet, diari..."
3,2025,"(1184, 20)","[source_year, source_file, source_sheet, diari..."


## Step 6: Build the Curated Dataset

After harmonizing and cleaning the yearly datasets, the dataframes were combined into one unified curated dataset.

At this stage, all datasets shared the same column structure, naming conventions, and normalized values, which made it possible to safely merge them into a single analysis-ready table.

Additional enrichment was also applied to improve the usability of the dataset for later analysis and API development. Examples of enrichment included approval indicators, sector categorization, and standardized education-related attributes.

The final curated dataset represents harmonized application-related information across multiple years in a consistent and structured format.

This dataset will later be used for validation, SQL database integration, and API development.

In [7]:
curated_df = pd.concat(harmonized_dfs.values(), ignore_index=True)
curated_df = curated_df.convert_dtypes()
curated_df = enrich_dataset(curated_df)

print("-------- CURATED DATASET OVERVIEW --------")
print(f"Rows: {curated_df.shape[0]}")
print(f"Columns: {curated_df.shape[1]}")

curated_df.head()

-------- CURATED DATASET OVERVIEW --------
Rows: 4921
Columns: 24


,source_year,source_file,source_sheet,diarienummer,utbildningsnamn,utbildningsomrade,beslut,decision_normalized,kommun,lan,...,utbildningsanordnare,huvudmannatyp,sun5_inriktning,sun5_inriktning_namn,seqf_niva,smalt_yrkesomrade,is_approved,education_length,sector_category,record_source
0,2022,resultat-ansokningsomgang-2022.xlsx,Tabell 3,MYH 2022/5458,.NET Cloud developer,Data/IT,Avslag,rejected,Stockholm,Stockholm,...,IT-Högskolan Stockholm AB,Privat,<NA>,<NA>,<NA>,<NA>,False,long,unknown,2022_Tabell 3
1,2022,resultat-ansokningsomgang-2022.xlsx,Tabell 3,MYH 2022/4695,.NET Developer,Data/IT,Avslag,rejected,Flera kommuner,Flera kommuner,...,KYH AB,Privat,<NA>,<NA>,<NA>,<NA>,False,long,unknown,2022_Tabell 3
2,2022,resultat-ansokningsomgang-2022.xlsx,Tabell 3,MYH 2022/4476,.NET Utvecklare,Data/IT,Beviljad,approved,Gävle,Gävleborg,...,Plushögskolan AB - Teknikhögskolan,Privat,<NA>,<NA>,<NA>,<NA>,True,long,unknown,2022_Tabell 3
3,2022,resultat-ansokningsomgang-2022.xlsx,Tabell 3,MYH 2022/4708,.NET Utvecklare,Data/IT,Avslag,rejected,Flera kommuner,Flera kommuner,...,Plushögskolan AB - Teknikhögskolan Mitt,Privat,<NA>,<NA>,<NA>,<NA>,False,long,unknown,2022_Tabell 3
4,2022,resultat-ansokningsomgang-2022.xlsx,Tabell 3,MYH 2022/5529,.NET-utvecklare,Data/IT,Avslag,rejected,Kungälv,Västra Götaland,...,Optimum Education AB,Privat,<NA>,<NA>,<NA>,<NA>,False,long,unknown,2022_Tabell 3


## Step 7: Validation and Quality Checks

After harmonization and merging, several validation and quality checks were performed to inspect the consistency and completeness of the curated dataset.

The validation focused on:

- dataset structure and data types
- missing values
- duplicate application records
- categorical consistency
- normalized analytical fields

Some missing values were expected due to historical schema differences between years.

For example, several SUN5-related classification fields were not available in the 2022 source files and therefore appear as missing values after harmonization.

In [8]:
print("--------- DATASET OVERVIEW ---------")
curated_df.info()

print("\n--------- MISSING VALUES ---------")
display(curated_df.isna().sum())

print("\n--------- DUPLICATE ROWS ---------")
print(curated_df.duplicated().sum())

print("\n--------- LOW-CARDINALITY COLUMN INSPECTION ---------")

inspect_columns = [
    "source_year",
    "decision_normalized",
    "studieform",
    "study_form_normalized",
    "education_length",
    "huvudmannatyp",
    "seqf_niva",
    "sector_category"
]

for col in inspect_columns:
    print(f"\n--- {col} ---")

    value_counts_df = curated_df[col].value_counts(dropna=False).reset_index()

    value_counts_df.columns = [col, "count"]

    display(value_counts_df)


--------- DATASET OVERVIEW ---------
<class 'pandas.DataFrame'>
RangeIndex: 4921 entries, 0 to 4920
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   source_year            4921 non-null   Int64  
 1   source_file            4921 non-null   string 
 2   source_sheet           4921 non-null   string 
 3   diarienummer           4921 non-null   string 
 4   utbildningsnamn        4921 non-null   string 
 5   utbildningsomrade      4921 non-null   string 
 6   beslut                 4921 non-null   string 
 7   decision_normalized    4921 non-null   string 
 8   kommun                 4921 non-null   string 
 9   lan                    4921 non-null   string 
 10  yh_poang               4921 non-null   Int64  
 11  studieform             4921 non-null   string 
 12  study_form_normalized  4921 non-null   string 
 13  studietakt_procent     4921 non-null   Int64  
 14  utbildningsanordnare   4921 no

source_year                 0
source_file                 0
source_sheet                0
diarienummer                0
utbildningsnamn             0
utbildningsomrade           0
beslut                      0
decision_normalized         0
kommun                      0
lan                         0
yh_poang                    0
studieform                  0
study_form_normalized       0
studietakt_procent          0
utbildningsanordnare        0
huvudmannatyp               0
sun5_inriktning          1207
sun5_inriktning_namn     1207
seqf_niva                1260
smalt_yrkesomrade        1207
is_approved                 0
education_length            0
sector_category             0
record_source               0
dtype: int64


--------- DUPLICATE ROWS ---------
0

--------- LOW-CARDINALITY COLUMN INSPECTION ---------

--- source_year ---


,source_year,count
0,2024,1272
1,2023,1258
2,2022,1207
3,2025,1184



--- decision_normalized ---


...

,huvudmannatyp,count
0,Privat,4073
1,Kommun,779
2,Region,60
3,Statlig,9



--- seqf_niva ---


,seqf_niva,count
0,5.0,3591
1,<NA>,1260
2,6.0,70



--- sector_category ---


,sector_category,count
0,unknown,1207
1,data_it,988
2,other,537
3,ekonomi_forsaljning,523
4,bygg_fastighet_vvs,472
5,teknik_industri,443
6,halso_sjukvard,308
7,pedagogik_socialt_arbete,173
8,transport,98
9,media_design_kultur,90


## Formal Validation Summary
In addition to exploratory inspection, a structured validation summary was created to identify potential data quality issues in a more standardized and reusable way.

In [9]:
validation_summary = build_validation_summary(curated_df)

display(validation_summary)

,check,affected_rows,severity
0,missing_diarienummer,0,critical
1,duplicate_diarienummer,0,warning
2,missing_utbildningsanordnare,0,warning
3,invalid_decision_values,0,warning
4,invalid_study_form_values,0,warning
5,missing_sun5_fields,1207,info


## Step 8: Export Curated Dataset

After validation and quality checks, the curated dataset was exported for further use.

The exported file represents the cleaned, harmonized, and combined dataset. It can later be loaded into a database and used as the data source for an API.

The dataset was exported as a CSV file to make it easy to inspect and reuse in the next part of the assignment.

In [10]:
CURATED_DATA_PATH.mkdir(
    parents=True,
    exist_ok=True,
)

curated_df.to_csv(
    CURATED_DATA_PATH / "curated_applications.csv",
    index=False,
)

## Reflection

Working with historical Excel files highlighted several common data engineering challenges, including inconsistent schemas, missing columns, and differences in formatting between years.

Defining a target schema before merging the datasets made the harmonization process more structured and easier to manage.

This assignment also demonstrated the importance of cleaning and validating data before loading it into a database or exposing it through an API.